# Dataset division

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_parquet("datasets/questoes_tratadas.parquet")

train_df, rem_df = train_test_split(df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(rem_df, test_size=0.5, random_state=42)

print(f"Train set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

train_df.to_parquet("datasets/divided/train.parquet", index=False)
val_df.to_parquet("datasets/divided/val.parquet", index=False)
test_df.to_parquet("datasets/divided/test.parquet", index=False)

# Grid search for classic hyperparams

In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

# ========= cria pastas =========
RESULTS_FOLDER = "results/grid-search"
os.makedirs(RESULTS_FOLDER, exist_ok=True)

CACHE_FOLDER = "cache_dir"
os.makedirs(CACHE_FOLDER, exist_ok=True)

# ========= dados =========
df_train = pd.read_parquet("datasets/divided/train.parquet")
df_val = pd.read_parquet("datasets/divided/val.parquet")

df_train = pd.concat([df_train, df_val], ignore_index=True)

# ========= split =========
X_train = df_train["texto_lem"]
y_train = df_train["materia"]

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

# ========= steps fixos =========
vectorizer = TfidfVectorizer(
    max_features=None,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.8,
)
selector = SelectKBest(chi2, k=20000)

test_models = {
    "LinearSVC": {
        "model": LinearSVC(random_state=42, max_iter=1000),
        "params": {
            "clf__C": np.logspace(-2, 1, 7),
            "clf__class_weight": ["balanced", None],
        },
    },
    "LogisticRegression": {
        "model": LogisticRegression(random_state=42, max_iter=1000),
        "params": {
            "clf__C": np.logspace(-2, 1, 7),
            "clf__class_weight": ["balanced", None],
            "clf__solver": ["lbfgs", "saga"],
        },
    },
    "RandomForest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "clf__n_estimators": [200, 500, 800],
            "clf__max_depth": [10, 20, 30],
            "clf__class_weight": ["balanced_subsample", "balanced", None],
        },
    },
    "NaiveBayes": {
        "model": MultinomialNB(),
        "params": {
            "clf__alpha": np.logspace(-2, 1, 9),
        },
    },
}

# ========= pipeline com cache =========
pipeline = Pipeline(
    [
        ("tfidf", vectorizer),
        ("chi2", selector),
        ("clf", LinearSVC()),
    ],
    memory=CACHE_FOLDER,
)


summary_rows = []

for nome, spec in test_models.items():
    print(f"\n=== Rodando GridSearchCV para {nome} ===")
    pipeline.set_params(clf=spec["model"])
    params = spec["params"]

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        verbose=2,
        refit=True,
        return_train_score=False,
        error_score="raise",
    )

    grid.fit(X_train, y_train)  # type: ignore

    # ---- salva resultados do Grid ----
    results = pd.DataFrame(grid.cv_results_)
    cols = ["rank_test_score", "mean_test_score"] + [
        c for c in results.columns if c.startswith("param_")
    ]
    results = results[cols].copy()
    results["mean_test_score"] = results["mean_test_score"].round(4)
    results = results.sort_values(by="rank_test_score")
    results_path = f"{RESULTS_FOLDER}/gridsearch_results_{nome.lower()}.csv"
    results.to_csv(results_path, index=False)


=== Rodando GridSearchCV para LinearSVC ===
Fitting 4 folds for each of 14 candidates, totalling 56 fits
[CV] END ............clf__C=0.01, clf__class_weight=balanced; total time=   5.1s
[CV] END ............clf__C=0.01, clf__class_weight=balanced; total time=   5.0s
[CV] END ................clf__C=0.01, clf__class_weight=None; total time=   4.9s
[CV] END ............clf__C=0.01, clf__class_weight=balanced; total time=   5.0s
[CV] END clf__C=0.03162277660168379, clf__class_weight=balanced; total time=   4.8s
[CV] END clf__C=0.03162277660168379, clf__class_weight=balanced; total time=   4.9s
[CV] END .clf__C=0.03162277660168379, clf__class_weight=None; total time=   4.9s
[CV] END clf__C=0.03162277660168379, clf__class_weight=balanced; total time=   5.0s
[CV] END ................clf__C=0.01, clf__class_weight=None; total time=   5.3s
[CV] END ............clf__C=0.01, clf__class_weight=balanced; total time=   5.8s
[CV] END clf__C=0.03162277660168379, clf__class_weight=balanced; total time

/home/murilob/codes/tcc/artigo-murilo/venv/lib/python3.11/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END ................clf__C=10.0, clf__class_weight=None; total time=   4.7s
[CV] END ............clf__C=10.0, clf__class_weight=balanced; total time=   4.9s
[CV] END ................clf__C=10.0, clf__class_weight=None; total time=   4.8s
[CV] END ................clf__C=10.0, clf__class_weight=None; total time=   4.4s
[CV] END ............clf__C=10.0, clf__class_weight=balanced; total time=   5.2s
[CV] END ............clf__C=10.0, clf__class_weight=balanced; total time=   5.0s
[CV] END ................clf__C=10.0, clf__class_weight=None; total time=   4.6s
[CV] END ............clf__C=10.0, clf__class_weight=balanced; total time=   5.2s

=== Rodando GridSearchCV para LogisticRegression ===
Fitting 4 folds for each of 28 candidates, totalling 112 fits
[CV] END clf__C=0.01, clf__class_weight=balanced, clf__solver=lbfgs; total time=   2.1s
[CV] END clf__C=0.01, clf__class_weight=balanced, clf__solver=lbfgs; total time=   2.1s
[CV] END clf__C=0.01, clf__class_weight=balanced, clf__solve

/home/murilob/codes/tcc/artigo-murilo/venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END clf__C=1.0, clf__class_weight=balanced, clf__solver=saga; total time= 1.8min


...

[CV] END clf__C=10.0, clf__class_weight=balanced, clf__solver=saga; total time= 1.3min


/home/murilob/codes/tcc/artigo-murilo/venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END clf__C=10.0, clf__class_weight=balanced, clf__solver=saga; total time= 1.2min

=== Rodando GridSearchCV para RandomForest ===
Fitting 4 folds for each of 27 candidates, totalling 108 fits
[CV] END clf__class_weight=balanced_subsample, clf__max_depth=10, clf__n_estimators=200; total time=   4.0s
[CV] END clf__class_weight=balanced_subsample, clf__max_depth=10, clf__n_estimators=200; total time=   4.3s
[CV] END clf__class_weight=balanced_subsample, clf__max_depth=10, clf__n_estimators=200; total time=   4.3s
[CV] END clf__class_weight=balanced_subsample, clf__max_depth=10, clf__n_estimators=200; total time=   4.6s
[CV] END clf__class_weight=balanced_subsample, clf__max_depth=10, clf__n_estimators=500; total time=   8.8s
[CV] END clf__class_weight=balanced_subsample, clf__max_depth=10, clf__n_estimators=500; total time=   8.8s
[CV] END clf__class_weight=balanced_subsample, clf__max_depth=10, clf__n_estimators=500; total time=   8.9s
[CV] END clf__class_weight=balanced_subsample, 

/home/murilob/codes/tcc/artigo-murilo/venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END clf__class_weight=balanced_subsample, clf__max_depth=10, clf__n_estimators=800; total time=  14.4s
[CV] END clf__class_weight=balanced_subsample, clf__max_depth=10, clf__n_estimators=800; total time=  15.4s
[CV] END clf__class_weight=balanced_subsample, clf__max_depth=10, clf__n_estimators=800; total time=  15.6s
[CV] END clf__class_weight=balanced_subsample, clf__max_depth=20, clf__n_estimators=500; total time=  21.7s
[CV] END clf__class_weight=balanced_subsample, clf__max_depth=30, clf__n_estimators=200; total time=  16.4s
[CV] END clf__class_weight=balanced_subsample, clf__max_depth=30, clf__n_estimators=200; total time=  17.0s
[CV] END clf__class_weight=balanced_subsample, clf__max_depth=20, clf__n_estimators=500; total time=  22.5s
[CV] END clf__class_weight=balanced_subsample, clf__max_depth=30, clf__n_estimators=200; total time=  17.2s
[CV] END clf__class_weight=balanced_subsample, clf__max_depth=20, clf__n_estimators=500; total time=  23.0s
[CV] END clf__class_weight=b

# One-stage Pipeline

## Classics

In [ ]:
import pandas as pd
import numpy as np
import os

from sklearn import clone
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

# ========= constants =========
CACHE_DIR = "cache_dir"
os.makedirs(CACHE_DIR, exist_ok=True)

RESULTS_DIR = "results/one-stage"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ========= data =========
df_train = pd.read_parquet(f"./datasets/divided/train.parquet")
df_val = pd.read_parquet(f"./datasets/divided/val.parquet")

df_train = pd.concat([df_train, df_val], ignore_index=True)

df_test = pd.read_parquet("datasets/divided/test.parquet")

X_train = df_train["texto_lem"]
y_train = df_train["topico"]

X_test = df_test["texto_lem"]
y_test = df_test["topico"]

# ========= define models =========
MODELS = {
    "svc": LinearSVC(C=0.316, class_weight="balanced", random_state=42),
    "lr": LogisticRegression(
        C=10.0, class_weight="balanced", solver="lbfgs", random_state=42, max_iter=1000
    ),
    "rf": RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced_subsample",
        max_depth=30,
        random_state=42,
        n_jobs=-1,
    ),
    "nb": MultinomialNB(alpha=0.01),
}

# ========= base pipeline =========
base_pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                max_features=None, ngram_range=(1, 2), min_df=3, max_df=0.8
            ),
        ),
        ("chi2", SelectKBest(chi2, k=20000)),
        ("clf", None),
    ],
    memory=CACHE_DIR,
)


# =========  helpers =========
def evaluate_model(modelo, nome_modelo, pipeline_base):
    print(f"\n=== Treinando e avaliando modelo: {nome_modelo} ===")
    # Clona o pipeline para garantir isolamento
    local_pipeline = clone(pipeline_base)
    local_pipeline.set_params(clf=modelo)

    # Treino e Predição
    local_pipeline.fit(X_train, y_train)
    y_pred = local_pipeline.predict(X_test)
    y_true = y_test

    print(classification_report(y_true, y_pred, digits=3, zero_division=0))

    # Retorna métricas
    return {
        "model": nome_modelo,
        "Accuracy": np.round(accuracy_score(y_true, y_pred), 3),
        "F1 Macro": np.round(f1_score(y_true, y_pred, average="macro"), 3),
        "F1 Weighted": np.round(f1_score(y_true, y_pred, average="weighted"), 3),
        "Recall Macro": np.round(recall_score(y_true, y_pred, average="macro"), 3),
        "Precision Macro": np.round(
            precision_score(y_true, y_pred, average="macro", zero_division=0), 3
        ),
        "preds": y_pred,
    }


# ========= evaluate models =========
results = []
for model_name, model in MODELS.items():
    results.append(evaluate_model(model, model_name, base_pipeline))

# ========= results consolidation =========
df_results = pd.DataFrame(results)

# metrics
df_metrics = df_results.drop(columns=["preds"])
df_metrics = df_metrics.sort_values(by="F1 Macro", ascending=False)

# predictions
dict_preds = {item["model"]: item["preds"] for item in results}
df_preds = pd.DataFrame(dict_preds)
df_preds["y_true"] = y_test

# ========= save to csv =========
df_metrics.to_csv(f"{RESULTS_DIR}/classics-results.csv", index=False)
df_preds.to_csv(f"{RESULTS_DIR}/classics-predictions.csv", index=False)


=== Treinando e avaliando modelo: svc ===
                                                    precision    recall  f1-score   support

                      América Latina Contemporânea      0.600     0.346     0.439        26
                            América Pré-Colombiana      0.538     1.000     0.700         7
              Análise Combinatória e Probabilidade      0.740     0.787     0.763        47
                                         Arcadismo      0.833     0.833     0.833         6
                    Aritmética e Matemática Básica      0.579     0.282     0.379        39
                           Arquitetura e Escultura      0.250     0.333     0.286         6
                               Arte Antiga e Média      0.000     0.000     0.000        11
                                Arte Contemporânea      0.105     0.167     0.129        12
                                      Arte Moderna      0.778     0.583     0.667        24
                Arte do Renascimento

## Transformers

### Tokenization

In [ ]:
import pandas as pd
import os
from transformers import BertTokenizer

if (
    not os.path.exists("./datasets/divided/test_tokenized.parquet")
    or not os.path.exists("./datasets/divided/train_tokenized.parquet")
    or not os.path.exists("./datasets/divided/val_tokenized.parquet")
):
    # Dados
    df_train = pd.read_parquet("./datasets/divided/train.parquet")
    df_val = pd.read_parquet("./datasets/divided/val.parquet")
    df_test = pd.read_parquet("./datasets/divided/test.parquet")

    dfs = {
        "train": df_train,
        "val": df_val,
        "test": df_test,
    }

    # Tokenização
    special_tokens = [
        "<num>",
        "<url>",
        "<delta>",
        "<graus>",
        "<raiz>",
        "<pi>",
        "<ohm>",
        "<lambda>",
        "<theta>",
        "<mu>",
        "<soma>",
    ]
    tokenizer = BertTokenizer.from_pretrained(
        "adalbertojunior/distilbert-portuguese-cased"
    )
    tokenizer.add_special_tokens({"additional_special_tokens": special_tokens})
    tokenizer.save_pretrained("./tokenizer-custom")

    for name, df in dfs.items():
        print(f"Tokenizando {name} set...")

        encodings = tokenizer(
            df["texto_clean_cased"].tolist(),
            truncation=True,
            padding="max_length",
            max_length=365,
        )

        df["input_ids"] = encodings["input_ids"]
        df["attention_mask"] = encodings["attention_mask"]

        df = df[["id", "materia", "topico", "input_ids", "attention_mask"]]

        df.to_parquet(
            f"./datasets/divided/{name}_tokenized.parquet",
            index=False,
        )

        print(f"Tokenização {name} concluída.")
else:
    print("Tokenized datasets already available. Skipping...")

### Training and Evaluation

In [1]:
import pandas as pd
import torch
import os
import numpy as np

import torch.nn.functional as F

from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from sklearn.utils import compute_class_weight

from transformers import BertForSequenceClassification, BertTokenizer
from transformers import TrainingArguments, Trainer
from transformers import EarlyStoppingCallback

# ======== constants =========
RESULTS_DIR = "./results/one-stage"
os.makedirs(RESULTS_DIR, exist_ok=True)


# ======== helpers ========
class QuestoesDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.input_ids = torch.tensor(df["input_ids"].tolist())
        self.attention_mask = torch.tensor(df["attention_mask"].tolist())
        self.labels = torch.tensor(df["label"].tolist())

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }

    def __len__(self):
        return len(self.labels)


def get_best_epoch_from_trainer(trainer: Trainer) -> float:
    metric_name = trainer.args.metric_for_best_model

    if not metric_name:
        metric_name = "loss"

    if not metric_name.startswith("eval_"):
        metric_name = f"eval_{metric_name}"

    best_metric_val = trainer.state.best_metric

    best_epoch = 0.0

    for log in trainer.state.log_history:
        if metric_name in log:
            if log[metric_name] == best_metric_val:
                best_epoch = log["epoch"]
                break

    return best_epoch


# ======== data ===========
train_df = pd.read_parquet(f"./datasets/divided/train_tokenized.parquet")
val_df = pd.read_parquet(f"./datasets/divided/val_tokenized.parquet")
test_df = pd.read_parquet(f"./datasets/divided/test_tokenized.parquet")
full_train = pd.concat([train_df, val_df], ignore_index=True)

train_df["label"] = train_df["topico"].astype("category").cat.codes

label2id = {
    v: k
    for k, v in dict(
        enumerate(train_df["topico"].astype("category").cat.categories)
    ).items()
}
id2label = dict(enumerate(train_df["topico"].astype("category").cat.categories))

val_df["label"] = val_df["topico"].map(label2id)
test_df["label"] = test_df["topico"].map(label2id)
full_train["label"] = full_train["topico"].map(label2id)



train_dataset = QuestoesDataset(train_df)
val_dataset = QuestoesDataset(val_df)
test_dataset = QuestoesDataset(test_df)
full_train_dataset = QuestoesDataset(full_train)

# ======== models ===========
MODELS = {
    # "distilbert": "adalbertojunior/distilbert-portuguese-cased",
    "bert": "neuralmind/bert-base-portuguese-cased",
}


# ======== custom metrics and loss ===========
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average="macro")
    f1_weighted = f1_score(labels, predictions, average="weighted")
    recall = recall_score(labels, predictions, average="macro", zero_division=0)
    precision = precision_score(labels, predictions, average="macro", zero_division=0)

    return {
        "accuracy": acc,
        "macro-precision": precision,
        "macro-recall": recall,
        "macro-f1": f1_macro,
        "weighted-f1": f1_weighted,
    }


# Cálculo dos pesos das classes
class_weights = compute_class_weight(
    class_weight="balanced", classes=np.unique(train_df["label"]), y=train_df["label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float)


def weighted_ce_loss(outputs, labels, num_items_in_batch: int):
    logits = outputs.logits
    weight = class_weights.to(logits.device)  # type: ignore
    return F.cross_entropy(logits, labels, weight=weight)


early_stopping_callback = EarlyStoppingCallback(
    early_stopping_threshold=0.005, early_stopping_patience=2
)

# ======== training args ===========
training_args = TrainingArguments(
    output_dir="./modelos-treinamento",  # Pasta para salvar o modelo
    eval_strategy="epoch",  # Avalia o modelo a cada época
    save_strategy="epoch",  # Salva o modelo a cada época
    save_total_limit=2,  # Mantém apenas os 2 últimos modelos salvos
    num_train_epochs=10,  # Número de épocas
    per_device_train_batch_size=12,  # Tamanho do batch de treino.
    per_device_eval_batch_size=24,  # Tamanho do batch de avaliação
    warmup_ratio=0.1,  # Passos de aquecimento do otimizador
    weight_decay=0.01,  # Regularização
    learning_rate=4e-5,  # Taxa de aprendizado
    load_best_model_at_end=True,  # Carrega o melhor modelo no final do treino
    metric_for_best_model="macro-f1",  # Usa a macro-f1 para decidir o melhor modelo
    logging_strategy="epoch",
)

# ======== preliminar training ===========
for model_name, model_path in MODELS.items():
    model = BertForSequenceClassification.from_pretrained(
        model_path,
        num_labels=len(id2label),
        id2label=id2label,
        label2id=label2id,
    )

    tokenizer = BertTokenizer.from_pretrained("./tokenizer-custom")
    model.resize_token_embeddings(len(tokenizer))

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        compute_loss_func=weighted_ce_loss,
        callbacks=[early_stopping_callback],
    )

    print(f"\n=== Treinamento preliminar modelo: {model_name} ===")

    trainer.train()

    best_epoch = get_best_epoch_from_trainer(trainer)

    print(f"Best epoch for {model_name}: {best_epoch}")

    # ======== final training (refit) ===========
    print(f"\n=== Treinamento final modelo (refit): {model_name} ===")

    final_training_args = TrainingArguments(
        output_dir="./modelos-treinamento-final",  # Pasta para salvar o modelo
        num_train_epochs=best_epoch,  # Número de épocas baseado no melhor da prévia # type: ignore
        per_device_train_batch_size=12,  # Tamanho do batch de treino.
        per_device_eval_batch_size=24,  # Tamanho do batch de avaliação
        learning_rate=4e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        save_strategy="no",  # Não precisa salvar checkpoints intermediários
        eval_strategy="no",  # Não avalia (não tem dataset de validação)
        logging_strategy="epoch",
    )

    final_trainer = Trainer(
        model=model,
        args=final_training_args,
        train_dataset=full_train_dataset,
        compute_metrics=compute_metrics,
        compute_loss_func=weighted_ce_loss,
    )

    final_trainer.train()

    # ======== evaluation ===========
    predictions = final_trainer.predict(test_dataset)
    predictions_label = np.argmax(predictions.predictions, axis=-1)

    metrics = compute_metrics((predictions.predictions, test_df["label"].to_numpy()))

    # ======== results consolidation ===========
    df_scores = pd.DataFrame(
        {name: value for name, value in metrics.items()}, index=[0]
    )
    print("\n=== Evaluation Scores ===")
    print(df_scores)

    df_preds = pd.DataFrame(
        {
            "y_true": test_df["label"],
            "y_pred": predictions_label,
        }
    )

    df_preds["y_true"] = df_preds["y_true"].map(id2label)
    df_preds["y_pred"] = df_preds["y_pred"].map(id2label)

    # ======== save to csv ===========
    df_scores.to_csv(
        f"{RESULTS_DIR}/{model_name}-results.csv",
        index=False,
    )

    df_preds.to_csv(
        f"{RESULTS_DIR}/{model_name}-predictions.csv",
        index=False,
    )

/home/murilob/codes/tcc/artigo-murilo/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`



=== Treinamento preliminar modelo: bert ===


Epoch,Training Loss,Validation Loss,Accuracy,Macro-precision,Macro-recall,Macro-f1,Weighted-f1
1,4.022600,2.702642,0.461715,0.344702,0.363661,0.311571,0.408287
2,2.095500,1.720837,0.559342,0.519328,0.562652,0.498414,0.541959
3,1.263600,1.496564,0.608729,0.579559,0.603775,0.567752,0.605035
4,0.821500,1.402821,0.631700,0.603129,0.621445,0.593988,0.633770
5,0.563400,1.464339,0.643951,0.597169,0.618132,0.593849,0.639617
6,0.405600,1.609466,0.654671,0.620745,0.631096,0.613440,0.652654
7,0.290900,1.681861,0.645865,0.611085,0.622259,0.605031,0.644533
8,0.230900,1.793250,0.656968,0.618032,0.631327,0.613213,0.656404


Best epoch for bert: 6.0

=== Treinamento final modelo (refit): bert ===


Step,Training Loss
1234,0.580400
2468,0.510400
3702,0.382500
4936,0.296800
6170,0.221900
7404,0.162900



=== Evaluation Scores ===
   accuracy  macro-precision  macro-recall  macro-f1  weighted-f1
0  0.666284         0.612292      0.609974  0.600179     0.664335


# Two-stage Pipeline

## First stage (subject)

### Classics

In [1]:
import pandas as pd
import numpy as np
import os

from sklearn import clone
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    classification_report,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

# ========= constants =========
CACHE_DIR = "cache_dir"
os.makedirs(CACHE_DIR, exist_ok=True)

RESULTS_DIR = "results/two-stage/subject-classification"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ========= data =========
df_train = pd.read_parquet(f"./datasets/divided/train.parquet")
df_val = pd.read_parquet(f"./datasets/divided/val.parquet")
df_test = pd.read_parquet("datasets/divided/test.parquet")

df_train = pd.concat([df_train, df_val], ignore_index=True)

X_train = df_train["texto_lem"]
y_train = df_train["materia"]

X_test = df_test["texto_lem"]
y_test = df_test["materia"]

# ========= define models =========
MODELS = {
    "svc": LinearSVC(C=0.316, class_weight="balanced", random_state=42),
    "lr": LogisticRegression(
        C=10.0, class_weight="balanced", solver="lbfgs", random_state=42, max_iter=1000
    ),
    "rf": RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced_subsample",
        max_depth=30,
        random_state=42,
        n_jobs=-1,
    ),
    "nb": MultinomialNB(alpha=0.01),
}

# ========= base pipeline =========
base_pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                max_features=None, ngram_range=(1, 2), min_df=3, max_df=0.8
            ),
        ),
        ("chi2", SelectKBest(chi2, k=20000)),
        ("clf", None),
    ],
    memory=CACHE_DIR,
)


# =========  helpers =========
def evaluate_model(modelo, nome_modelo, pipeline_base):
    print(f"\n=== Treinando e avaliando modelo: {nome_modelo} ===")
    # Clona o pipeline para garantir isolamento
    local_pipeline = clone(pipeline_base)
    local_pipeline.set_params(clf=modelo)

    # Treino e Predição
    local_pipeline.fit(X_train, y_train)
    y_pred = local_pipeline.predict(X_test)
    y_true = y_test

    print(classification_report(y_true, y_pred, digits=3, zero_division=0))

    # Retorna métricas
    return {
        "model": nome_modelo,
        "Accuracy": np.round(accuracy_score(y_true, y_pred), 3),
        "F1 Macro": np.round(f1_score(y_true, y_pred, average="macro"), 3),
        "F1 Weighted": np.round(f1_score(y_true, y_pred, average="weighted"), 3),
        "Recall Macro": np.round(recall_score(y_true, y_pred, average="macro"), 3),
        "Precision Macro": np.round(
            precision_score(y_true, y_pred, average="macro", zero_division=0), 3
        ),
        "preds": y_pred,
    }


# ========= evaluate models =========
results = []
for model_name, model in MODELS.items():
    results.append(evaluate_model(model, model_name, base_pipeline))

# ========= results consolidation =========
df_results = pd.DataFrame(results)

# metrics
df_metrics = df_results.drop(columns=["preds"])
df_metrics = df_metrics.sort_values(by="F1 Macro", ascending=False)

# predictions
dict_preds = {item["model"]: item["preds"] for item in results}
df_preds = pd.DataFrame(dict_preds)
df_preds["y_true"] = y_test

# ========= save to csv =========
df_metrics.to_csv(f"{RESULTS_DIR}/classics-results.csv", index=False)
df_preds.to_csv(f"{RESULTS_DIR}/classics-predictions.csv", index=False)


=== Treinando e avaliando modelo: svc ===
              precision    recall  f1-score   support

       artes      0.750     0.785     0.767       107
    biologia      0.947     0.907     0.926       451
   filosofia      0.940     0.838     0.886       130
      fisica      0.905     0.914     0.910       292
   geografia      0.714     0.769     0.741        91
    historia      0.929     0.949     0.939       509
     idiomas      0.970     1.000     0.985        32
  literatura      0.899     0.893     0.896       169
  matematica      0.921     0.945     0.933       397
   portugues      0.737     0.771     0.753       109
     quimica      0.906     0.900     0.903       280
  sociologia      0.600     0.522     0.558        46

    accuracy                          0.896      2613
   macro avg      0.851     0.849     0.850      2613
weighted avg      0.896     0.896     0.896      2613


=== Treinando e avaliando modelo: lr ===
              precision    recall  f1-score   su

### Transformers

In [ ]:
import torch
import os

import pandas as pd
import numpy as np

import torch.nn.functional as F

from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from sklearn.utils import compute_class_weight

from transformers import BertForSequenceClassification, BertTokenizer
from transformers import TrainingArguments, Trainer
from transformers import EarlyStoppingCallback

# ======== constants =========
RESULTS_DIR = "results/two-stage/subject-classification"
os.makedirs(RESULTS_DIR, exist_ok=True)


# ======== helpers ========
class QuestoesDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.input_ids = torch.tensor(df["input_ids"].tolist())
        self.attention_mask = torch.tensor(df["attention_mask"].tolist())
        self.labels = torch.tensor(df["label"].tolist())

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }

    def __len__(self):
        return len(self.labels)


def get_best_epoch_from_trainer(trainer: Trainer) -> float:
    metric_name = trainer.args.metric_for_best_model

    if not metric_name:
        metric_name = "loss"

    if not metric_name.startswith("eval_"):
        metric_name = f"eval_{metric_name}"

    best_metric_val = trainer.state.best_metric

    best_epoch = 0.0

    for log in trainer.state.log_history:
        if metric_name in log:
            if log[metric_name] == best_metric_val:
                best_epoch = log["epoch"]
                break

    return best_epoch


# ======== data ===========
train_df = pd.read_parquet(f"./datasets/divided/train_tokenized.parquet")
val_df = pd.read_parquet(f"./datasets/divided/val_tokenized.parquet")
test_df = pd.read_parquet(f"./datasets/divided/test_tokenized.parquet")

train_df["label"] = train_df["materia"].astype("category").cat.codes

label2id = {
    v: k
    for k, v in dict(
        enumerate(train_df["materia"].astype("category").cat.categories)
    ).items()
}
id2label = dict(enumerate(train_df["materia"].astype("category").cat.categories))

val_df["label"] = val_df["materia"].map(label2id)
test_df["label"] = test_df["materia"].map(label2id)

full_train = pd.concat([train_df, val_df], ignore_index=True)


train_dataset = QuestoesDataset(train_df)
val_dataset = QuestoesDataset(val_df)
test_dataset = QuestoesDataset(test_df)
full_train_dataset = QuestoesDataset(full_train)

# ======== models ===========
MODELS = {
    "distilbert": "adalbertojunior/distilbert-portuguese-cased",
    "bert": "neuralmind/bert-base-portuguese-cased",
}


# ======== custom metrics and loss ===========
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average="macro")
    f1_weighted = f1_score(labels, predictions, average="weighted")
    recall = recall_score(labels, predictions, average="macro", zero_division=0)
    precision = precision_score(labels, predictions, average="macro", zero_division=0)

    return {
        "accuracy": acc,
        "macro-precision": precision,
        "macro-recall": recall,
        "macro-f1": f1_macro,
        "weighted-f1": f1_weighted,
    }


# Cálculo dos pesos das classes
class_weights = compute_class_weight(
    class_weight="balanced", classes=np.unique(train_df["label"]), y=train_df["label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float)


def weighted_ce_loss(outputs, labels, num_items_in_batch: int):
    logits = outputs.logits
    weight = class_weights.to(logits.device)  # type: ignore
    return F.cross_entropy(logits, labels, weight=weight)


early_stopping_callback = EarlyStoppingCallback(
    early_stopping_threshold=0.005, early_stopping_patience=2
)

# ======== training args ===========
training_args = TrainingArguments(
    output_dir="./modelos-treinamento",  # Pasta para salvar o modelo
    eval_strategy="epoch",  # Avalia o modelo a cada época
    save_strategy="epoch",  # Salva o modelo a cada época
    save_total_limit=2,  # Mantém apenas os 2 últimos modelos salvos
    num_train_epochs=10,  # Número de épocas
    per_device_train_batch_size=12,  # Tamanho do batch de treino.
    per_device_eval_batch_size=24,  # Tamanho do batch de avaliação
    warmup_ratio=0.1,  # Passos de aquecimento do otimizador
    weight_decay=0.01,  # Regularização
    learning_rate=4e-5,  # Taxa de aprendizado
    logging_strategy="epoch",
    load_best_model_at_end=True,  # Carrega o melhor modelo no final do treino
    metric_for_best_model="macro-f1",  # Usa a macro-f1 para decidir o melhor modelo
)

# ======== preliminar training ===========
for model_name, model_path in MODELS.items():
    model = BertForSequenceClassification.from_pretrained(
        model_path,
        num_labels=len(id2label),
        id2label=id2label,
        label2id=label2id,
    )

    tokenizer = BertTokenizer.from_pretrained("./tokenizer-custom")
    model.resize_token_embeddings(len(tokenizer))

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        compute_loss_func=weighted_ce_loss,
        callbacks=[early_stopping_callback],
    )

    print(f"\n=== Treinamento preliminar modelo: {model_name} ===")

    trainer.train()

    best_epoch = get_best_epoch_from_trainer(trainer)

    print(f"Best epoch for {model_name}: {best_epoch}")

    # ======== final training (refit) ===========
    print(f"\n=== Treinamento final modelo (refit): {model_name} ===")

    final_training_args = TrainingArguments(
        output_dir="./modelos-treinamento-final",  # Pasta para salvar o modelo
        num_train_epochs=best_epoch,  # Número de épocas baseado no melhor da prévia # type: ignore
        per_device_train_batch_size=12,  # Tamanho do batch de treino.
        per_device_eval_batch_size=24,  # Tamanho do batch de avaliação
        learning_rate=4e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        save_strategy="no",  # Não precisa salvar checkpoints intermediários
        eval_strategy="no",  # Não avalia (não tem dataset de validação)
        logging_strategy="epoch",
    )

    final_trainer = Trainer(
        model=model,
        args=final_training_args,
        train_dataset=full_train_dataset,
        compute_metrics=compute_metrics,
        compute_loss_func=weighted_ce_loss,
    )

    final_trainer.train()

    # ======== evaluation ===========
    predictions = final_trainer.predict(test_dataset)
    predictions_label = np.argmax(predictions.predictions, axis=-1)

    metrics = compute_metrics((predictions.predictions, test_df["label"].to_numpy()))

    # ======== results consolidation ===========
    df_scores = pd.DataFrame(
        {name: value for name, value in metrics.items()}, index=[0]
    )
    print("\n=== Evaluation Scores ===")
    print(df_scores)

    df_preds = pd.DataFrame(
        {
            "y_true": test_df["label"],
            "y_pred": predictions_label,
        }
    )

    df_preds["y_true"] = df_preds["y_true"].map(id2label)
    df_preds["y_pred"] = df_preds["y_pred"].map(id2label)

    # ======== save to csv ===========
    df_scores.to_csv(
        f"{RESULTS_DIR}/{model_name}-results.csv",
        index=False,
    )

    df_preds.to_csv(
        f"{RESULTS_DIR}/{model_name}-predictions.csv",
        index=False,
    )

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at adalbertojunior/distilbert-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== Treinamento preliminar modelo: distilbert ===


Epoch,Training Loss,Validation Loss,Accuracy,Macro-precision,Macro-recall,Macro-f1,Weighted-f1
1,0.578200,0.551181,0.868683,0.802658,0.831714,0.804150,0.875176
2,0.451600,0.520403,0.890123,0.836166,0.855358,0.839259,0.893690
3,0.305700,0.616194,0.873660,0.794775,0.850303,0.808971,0.879714
4,0.162300,0.724675,0.905819,0.851483,0.850255,0.848533,0.905381
5,0.112900,0.845015,0.906968,0.861438,0.837900,0.847358,0.905592
6,0.144400,0.845593,0.908116,0.849510,0.844359,0.844795,0.908313



=== Treinamento preliminar concluído ===
Best epoch for distilbert: 6.0

=== Treinamento final modelo (refit): distilbert ===


Step,Training Loss
100,0.231600
200,0.184500
300,0.122400
400,0.256300
500,0.158900
600,0.267100
700,0.270700
800,0.387500
900,0.270500
1000,0.253700


...

Epoch,Training Loss,Validation Loss,Accuracy,Macro-precision,Macro-recall,Macro-f1,Weighted-f1
1,0.531700,0.555026,0.873277,0.802356,0.843213,0.810858,0.879311
2,0.451900,0.548769,0.898162,0.839511,0.862919,0.843511,0.902860
3,0.264400,0.556560,0.897779,0.826353,0.858679,0.837472,0.901135
4,0.153700,0.702223,0.920368,0.877693,0.864215,0.869888,0.919666
5,0.141100,0.705927,0.911945,0.853843,0.856814,0.852927,0.912383
6,0.068600,0.793116,0.915008,0.862962,0.854216,0.856735,0.915074



=== Treinamento preliminar concluído ===
Best epoch for bert: 6.0

=== Treinamento final modelo (refit): bert ===


Step,Training Loss
100,0.211200
200,0.192200
300,0.109000
400,0.268900
500,0.179400
600,0.302200
700,0.293100
800,0.332100
900,0.272300
1000,0.311100



=== Treinamento final concluído ===



=== Evaluation Scores ===
   accuracy  macro-precision  macro-recall  macro-f1  weighted-f1
0  0.916954         0.887843      0.869741  0.877077     0.916234


## Second stage (topic)

### Classics

In [ ]:
import os
import pandas as pd

from concurrent.futures import ProcessPoolExecutor
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectPercentile, chi2
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
)

# ======== constants =========
RESULTS_DIR = "results/two-stage/topic-classification"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ======== data ===========
df_train = pd.read_parquet(f"./datasets/divided/train.parquet")
df_val = pd.read_parquet(f"./datasets/divided/val.parquet")
df_test = pd.read_parquet("datasets/divided/test.parquet")

df_train = pd.concat([df_train, df_val], ignore_index=True)

bert_preds = pd.read_csv(
    "results/two-stage/subject-classification/bert-predictions.csv"
)

df_test["bert_pred_subject"] = bert_preds["y_pred"]

# ======== models and pipelines ===========
models = {
    "svc": LinearSVC(C=0.316, class_weight="balanced", random_state=42),
    "lr": LogisticRegression(
        C=10.0, class_weight="balanced", solver="lbfgs", random_state=42, max_iter=1000
    ),
    "rf": RandomForestClassifier(
        n_estimators=800,
        class_weight="balanced_subsample",
        max_depth=30,
        random_state=42,
    ),
    "nb": MultinomialNB(alpha=0.01),
}

vectorizer = TfidfVectorizer(
    max_features=None,
    ngram_range=(1, 3),
    min_df=3,
    max_df=0.8,
)

selector = SelectPercentile(chi2, percentile=30)

base_pipeline = Pipeline(
    [
        ("tfidf", vectorizer),
        ("chi2", selector),
        ("clf", None),
    ]
)

subjects = df_train["materia"].unique()


# ======== helpers ===========
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
        "precision_macro": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }


def flatten_metrics(metrics_dict, prefix):
    return {f"{prefix}_{k}": v for k, v in metrics_dict.items()}


def train_and_preds(name, model, materia):
    pipeline = clone(base_pipeline)
    pipeline.set_params(clf=model)

    # --- 1. Preparação dos Dados ---
    # Treino (filtrado por matéria)
    mask_train = df_train["materia"] == materia
    x_train = df_train[mask_train]["texto_lem"]
    y_train = df_train[mask_train]["topico"]

    # Teste Gold (Gabarito: Matéria Real == materia atual)
    mask_gold = df_test["materia"] == materia
    x_test_gold = df_test[mask_gold]["texto_lem"]
    y_test_gold = df_test[mask_gold]["topico"]
    ids_gold = df_test[mask_gold]["id"]  # <--- GUARDAMOS OS ÍNDICES ORIGINAIS

    # Teste Real (Pipeline: Predição do BERT == materia atual)
    mask_real = df_test["bert_pred_subject"] == materia
    x_test_real = df_test[mask_real]["texto_lem"]
    ids_real = df_test[mask_real]["id"]  # <--- GUARDAMOS OS ÍNDICES ORIGINAIS

    # --- 2. Treinamento ---
    pipeline.fit(x_train, y_train)

    # --- 3. Inferência ---

    # 3.2 Avaliação Gold
    df_preds_gold = pd.DataFrame()
    df_preds_gold["id"] = ids_gold
    df_preds_gold["subject_true"] = materia
    df_preds_gold["y_true"] = y_test_gold
    df_preds_gold["y_pred_gold"] = pipeline.predict(x_test_gold)

    # 3.1 Avaliação Real
    df_preds_real = pd.DataFrame()
    df_preds_real["id"] = ids_real
    df_preds_real["subject_bert"] = materia
    df_preds_real["y_pred_real"] = pipeline.predict(x_test_real)

    # Merge DFs
    df_preds_merged = pd.merge(
        df_preds_gold,
        df_preds_real,
        on=["id"],
        how="outer",
    )
    df_preds_merged["model"] = name

    return df_preds_merged


# ======== training execution ===========
tasks = []
preds_dfs = []

with ProcessPoolExecutor() as executor:
    # Dispara as tarefas
    for name, model in models.items():
        for subject in subjects:
            tasks.append(executor.submit(train_and_preds, name, model, subject))

    # Coleta os resultados
    for future in tasks:
        res = future.result()
        if res is not None:
            preds = res
            preds_dfs.append(preds)

# ======== saving ===========

df_final_results = pd.concat(preds_dfs, ignore_index=True)
df_consolidado = df_final_results.groupby(["id", "model"], as_index=False).first()
df_consolidado.to_csv(
    f"{RESULTS_DIR}/classics-predictions.csv",
    index=False,
)

print("Processamento concluído e arquivos salvos!")

Processamento concluído e arquivos salvos!


### Transformers

In [ ]:
import torch
import os

import pandas as pd
import numpy as np

import torch.nn.functional as F

from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from sklearn.utils import compute_class_weight

from transformers import BertForSequenceClassification, BertTokenizer
from transformers import TrainingArguments, Trainer
from transformers import EarlyStoppingCallback

# ======== constants =========
RESULTS_DIR = "results/two-stage/topic-classification"
os.makedirs(RESULTS_DIR, exist_ok=True)


# ======== helpers ========
class QuestoesDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.input_ids = torch.tensor(df["input_ids"].tolist())
        self.attention_mask = torch.tensor(df["attention_mask"].tolist())
        self.labels = torch.tensor(df["label"].tolist())

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }

    def __len__(self):
        return len(self.labels)


def get_best_epoch_from_trainer(trainer: Trainer) -> float:
    metric_name = trainer.args.metric_for_best_model

    if not metric_name:
        metric_name = "loss"

    if not metric_name.startswith("eval_"):
        metric_name = f"eval_{metric_name}"

    best_metric_val = trainer.state.best_metric

    best_epoch = 0.0

    for log in trainer.state.log_history:
        if metric_name in log:
            if log[metric_name] == best_metric_val:
                best_epoch = log["epoch"]
                break

    return best_epoch


# ======== data ===========
df_train = pd.read_parquet(f"./datasets/divided/train_tokenized.parquet")
df_val = pd.read_parquet(f"./datasets/divided/val_tokenized.parquet")

df_test = pd.read_parquet(f"./datasets/divided/test_tokenized.parquet")
bert_preds = pd.read_csv(
    "results/two-stage/subject-classification/bert-predictions.csv"
)
df_test["bert_pred_subject"] = bert_preds["y_pred"]

# ======== models ===========
MODELS = {
    "distilbert": "adalbertojunior/distilbert-portuguese-cased",
    "bert": "neuralmind/bert-base-portuguese-cased",
}


# ======== custom metrics and loss ===========
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average="macro")
    f1_weighted = f1_score(labels, predictions, average="weighted")
    recall = recall_score(labels, predictions, average="macro", zero_division=0)
    precision = precision_score(labels, predictions, average="macro", zero_division=0)

    return {
        "accuracy": acc,
        "macro-precision": precision,
        "macro-recall": recall,
        "macro-f1": f1_macro,
        "weighted-f1": f1_weighted,
    }


early_stopping_callback = EarlyStoppingCallback(
    early_stopping_threshold=0.005, early_stopping_patience=4
)

# ======== training args ===========
training_args = TrainingArguments(
    output_dir="./modelos-treinamento",  # Pasta para salvar o modelo
    eval_strategy="epoch",  # Avalia o modelo a cada época
    save_strategy="epoch",  # Salva o modelo a cada época
    save_total_limit=2,  # Mantém apenas os 2 últimos modelos salvos
    num_train_epochs=20,  # Número de épocas
    per_device_train_batch_size=12,  # Tamanho do batch de treino.
    per_device_eval_batch_size=24,  # Tamanho do batch de avaliação
    warmup_ratio=0.1,  # Passos de aquecimento do otimizador
    weight_decay=0.01,  # Regularização
    learning_rate=3e-5,  # Taxa de aprendizado
    logging_strategy="epoch",
    load_best_model_at_end=True,  # Carrega o melhor modelo no final do treino
    metric_for_best_model="macro-f1",  # Usa a macro-f1 para decidir o melhor modelo
)

tokenizer = BertTokenizer.from_pretrained("./tokenizer-custom")

subjects = df_train["materia"].unique()


# ======== training and prediction function ===========
def train_preds(subject: str):
    # Filtra treino e validação pela matéria atual
    masked_train = df_train[df_train["materia"] == subject].copy()
    masked_val = df_val[df_val["materia"] == subject].copy()

    label2id = {
        v: k
        for k, v in dict(
            enumerate(masked_train["topico"].astype("category").cat.categories)
        ).items()
    }
    id2label = dict(enumerate(masked_train["topico"].astype("category").cat.categories))

    masked_train["label"] = masked_train["topico"].map(label2id)
    masked_val["label"] = masked_val["topico"].map(label2id)

    train_dataset = QuestoesDataset(masked_train)
    val_dataset = QuestoesDataset(masked_val)

    # peso das classes específico para a matéria
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(masked_train["label"]),
        y=masked_train["label"],
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float)

    def weighted_ce_loss(outputs, labels, num_items_in_batch: int):
        logits = outputs.logits
        weight = class_weights.to(logits.device)  # type: ignore
        return F.cross_entropy(logits, labels, weight=weight)

    # cria modelo novo para cada matéria (sem compartilhar pesos de saída)
    model = BertForSequenceClassification.from_pretrained(
        model_path,
        num_labels=len(id2label),
        id2label=id2label,
        label2id=label2id,
    )
    model.resize_token_embeddings(len(tokenizer))

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        compute_loss_func=weighted_ce_loss,
        callbacks=[early_stopping_callback],
    )

    print(f"\n=== Treinamento preliminar: {subject} ===")
    trainer.train()

    best_epoch = get_best_epoch_from_trainer(trainer)
    print(f"Best epoch for {model_name}: {best_epoch}")

    # ======== final training (refit) ===========
    print(f"\n=== Treinamento final modelo (refit): {model_name} ===")

    masked_full_train = pd.concat([masked_train, masked_val], ignore_index=True)
    full_train_dataset = QuestoesDataset(masked_full_train)

    # recalcula peso das classes específico para a matéria
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(masked_full_train["label"]),
        y=masked_full_train["label"],
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float)

    # cria modelo novo para o treino final
    model = BertForSequenceClassification.from_pretrained(
        model_path,
        num_labels=len(id2label),
        id2label=id2label,
        label2id=label2id,
    )
    model.resize_token_embeddings(len(tokenizer))

    final_training_args = TrainingArguments(
        output_dir="./modelos-treinamento-final",  # Pasta para salvar o modelo
        num_train_epochs=best_epoch,  # Número de épocas baseado no melhor da prévia # type: ignore
        per_device_train_batch_size=12,  # Tamanho do batch de treino.
        per_device_eval_batch_size=24,  # Tamanho do batch de avaliação
        learning_rate=3e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        save_strategy="no",  # Não precisa salvar checkpoints intermediários
        eval_strategy="no",  # Não avalia (não tem dataset de validação)
        logging_strategy="epoch",
    )

    final_trainer = Trainer(
        model=model,
        args=final_training_args,
        train_dataset=full_train_dataset,
        compute_metrics=compute_metrics,
        compute_loss_func=weighted_ce_loss,
    )

    final_trainer.train()

    # ======== inferencia ===========

    # Teste Gold (filtrado pela matéria atual)
    masked_test_gold = df_test[df_test["materia"] == subject].copy()
    masked_test_gold["label"] = masked_test_gold["topico"].map(label2id)
    ids_gold = masked_test_gold["id"]
    true_labels = masked_test_gold["topico"].tolist()

    test_dataset = QuestoesDataset(masked_test_gold)

    predictions = final_trainer.predict(test_dataset)
    predictions = np.argmax(predictions.predictions, axis=-1)
    predictions = [id2label[pred] for pred in predictions]

    df_preds_gold = pd.DataFrame()
    df_preds_gold["id"] = ids_gold
    df_preds_gold["subject_true"] = subject
    df_preds_gold["y_true"] = true_labels
    df_preds_gold["y_pred_gold"] = predictions

    # Teste Real (filtrado pela predição do BERT na matéria atual)
    masked_test_real = df_test[df_test["bert_pred_subject"] == subject].copy()
    masked_test_real["label"] = masked_test_real["topico"].map(label2id)
    masked_test_real["label"] = masked_test_real["label"].fillna(0).astype(int)
    ids_real = masked_test_real["id"]

    test_dataset = QuestoesDataset(masked_test_real)

    predictions = final_trainer.predict(test_dataset)
    predictions = np.argmax(predictions.predictions, axis=-1)
    predictions = [id2label[pred] for pred in predictions]

    df_preds_real = pd.DataFrame()
    df_preds_real["id"] = ids_real
    df_preds_real["subject_bert"] = subject
    df_preds_real["y_pred_real"] = predictions

    # Merge DFs
    df_preds_merged = pd.merge(
        df_preds_gold,
        df_preds_real,
        on=["id"],
        how="outer",
    )
    df_preds_merged["model"] = model_name

    return df_preds_merged


preds_dfs = []
for model_name, model_path in MODELS.items():
    print(f"\n=== Processando modelo: {model_name} ===")

    for subject in subjects:
        preds = train_preds(subject)
        preds_dfs.append(preds)

# ======== saving ===========
df_final_results = pd.concat(preds_dfs, ignore_index=True)
df_consolidado = df_final_results.groupby(["id", "model"], as_index=False).first()
df_consolidado.to_csv(
    f"{RESULTS_DIR}/transformers-predictions.csv",
    index=False,
)

print("Processamento concluído e arquivos salvos!")

/home/murilob/codes/tcc/artigo-murilo/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



=== Processando modelo: distilbert ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at adalbertojunior/distilbert-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`



=== Treinamento preliminar: geografia ===


Epoch,Training Loss,Validation Loss,Accuracy,Macro-precision,Macro-recall,Macro-f1,Weighted-f1
1,2.083500,2.058205,0.098765,0.136218,0.142857,0.051838,0.035839
2,1.977700,1.765651,0.493827,0.621632,0.489744,0.462501,0.480188
3,1.420500,1.331169,0.493827,0.496264,0.488255,0.481002,0.496356
4,1.022000,1.211778,0.493827,0.580043,0.581319,0.545968,0.496746
5,0.755100,1.224950,0.518519,0.596828,0.573993,0.539534,0.511532
6,0.593000,1.337398,0.518519,0.575699,0.581777,0.549729,0.516183
7,0.449000,1.297754,0.629630,0.667037,0.646154,0.636566,0.627839
8,0.310300,1.382885,0.518519,0.626234,0.589606,0.570729,0.521402
9,0.272800,1.396734,0.543210,0.543498,0.556296,0.543786,0.553313
10,0.159800,1.481801,0.530864,0.559501,0.563233,0.549784,0.549254


...

Epoch,Training Loss,Validation Loss,Accuracy,Macro-precision,Macro-recall,Macro-f1,Weighted-f1
1,0.605300,0.404459,1.000000,1.000000,1.000000,1.000000,1.000000
2,0.230700,0.056432,1.000000,1.000000,1.000000,1.000000,1.000000
3,0.053400,0.069938,0.965517,0.964286,0.968750,0.965352,0.965600
4,0.008500,0.031433,0.965517,0.964286,0.968750,0.965352,0.965600
5,0.002800,0.024797,0.965517,0.964286,0.968750,0.965352,0.965600


Best epoch for bert: 1.0

=== Treinamento final modelo (refit): bert ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
15,0.437900


Processamento concluído e arquivos salvos!
